In [ ]:
%%writefile cleaning_pipeline.py
import numpy as np
import pandas as pd


def clean_agent_data(txns: pd.DataFrame, agents: pd.DataFrame) -> tuple[pd.DataFrame, list[dict]]:
    """
    Clean and validate mobile-money agent transactions and merge with the agent register.

    Steps performed:
    1. Deduplicate exact/repeat transactions on `txn_id` (keep first).
    2. Standardise `txn_type` to 'Deposit' and 'Withdrawal'.
    3. Convert `amount` to numeric (coercing invalid tokens like 'N/A' to NaN) and
       `txn_date` to ISO datetime / string format (`YYYY-MM-DD`).
    4. Flag and impute missing `amount` values using median by `txn_type` so
       transaction counts and volume distributions remain unbiased.
    5. Flag statistical outliers in `amount` using the IQR rule (Q3 + 1.5 * IQR)
       without deleting valid high-value transactions.
    6. Left-join `agents` on `agent_id`, labelling unmatched agents ('AG099') as 'Unknown'.

    Parameters
    ----------
    txns : pd.DataFrame
        Raw transaction extract (`agent_transactions.csv`).
    agents : pd.DataFrame
        Agent register (`agents.csv`).

    Returns
    -------
    tuple[pd.DataFrame, list[dict]]
        (clean_df, log) where `clean_df` is the cleaned, merged DataFrame and `log`
        is a structured list of cleaning actions and row counts for governance review.
    """
    df = txns.copy()
    agents_clean = agents.copy()
    log: list[dict] = []

    raw_rows = len(df)
    log.append({"step": "0_load", "action": "Loaded raw transactions", "rows_affected": raw_rows, "remaining_rows": raw_rows})

    # 1. Uniqueness: remove duplicate txn_id rows
    dup_count = int(df.duplicated(subset=["txn_id"]).sum())
    df = df.drop_duplicates(subset=["txn_id"], keep="first").reset_index(drop=True)
    log.append({
        "step": "1_deduplicate",
        "action": "Dropped duplicate txn_id rows (kept first occurrence)",
        "rows_affected": dup_count,
        "remaining_rows": len(df),
    })

    # 2. Consistency: standardise txn_type
    raw_type = df["txn_type"].astype(str)
    clean_type = raw_type.str.strip().str.title()
    non_standard_count = int((raw_type != clean_type).sum())
    df["txn_type"] = clean_type
    valid_types = {"Deposit", "Withdrawal"}
    invalid_type_count = int((~df["txn_type"].isin(valid_types)).sum())
    log.append({
        "step": "2_standardise_txn_type",
        "action": "Trimmed whitespace and applied title case to txn_type ('Deposit', 'Withdrawal')",
        "rows_affected": non_standard_count,
        "remaining_rows": len(df),
    })
    if invalid_type_count > 0:
        log.append({
            "step": "2b_unexpected_txn_type",
            "action": "Found unexpected txn_type values outside Deposit/Withdrawal",
            "rows_affected": invalid_type_count,
            "remaining_rows": len(df),
        })

    # 3. Validity & types: numeric amount and ISO txn_date
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

    raw_date_str = df["txn_date"].astype(str).str.strip()
    slash_dates = int(raw_date_str.str.contains("/", regex=False).sum())
    parsed_dates = pd.to_datetime(raw_date_str, format="mixed", errors="coerce")
    df["txn_date"] = parsed_dates.dt.strftime("%Y-%m-%d")
    log.append({
        "step": "3_parse_types",
        "action": "Converted amount to float64 and normalised mixed txn_date formats to ISO YYYY-MM-DD",
        "rows_affected": slash_dates,
        "remaining_rows": len(df),
    })

    # 4. Completeness: flag and impute missing amounts using median by txn_type
    df["amount_imputed"] = df["amount"].isna()
    imputed_count = int(df["amount_imputed"].sum())
    df["amount"] = df["amount"].fillna(df.groupby("txn_type")["amount"].transform("median"))
    log.append({
        "step": "4_impute_amount",
        "action": "Flagged missing/N/A amounts in amount_imputed and imputed with median per txn_type",
        "rows_affected": imputed_count,
        "remaining_rows": len(df),
    })

    # 5. Outliers: IQR rule on amount (flag, do not drop)
    q1 = df["amount"].quantile(0.25)
    q3 = df["amount"].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr
    lower_bound = q1 - 1.5 * iqr
    df["is_outlier"] = (df["amount"] < lower_bound) | (df["amount"] > upper_bound)
    outlier_count = int(df["is_outlier"].sum())
    log.append({
        "step": "5_flag_outliers",
        "action": f"Flagged IQR outliers in is_outlier (bounds: [{lower_bound:.2f}, {upper_bound:.2f}])",
        "rows_affected": outlier_count,
        "remaining_rows": len(df),
    })

    # 6. Referential integrity: left merge with agents register
    agents_clean["agent_id"] = agents_clean["agent_id"].astype(str).str.strip()
    df["agent_id"] = df["agent_id"].astype(str).str.strip()

    merged = pd.merge(
        df,
        agents_clean[["agent_id", "county", "onboarded"]],
        on="agent_id",
        how="left",
    )
    unknown_agent_count = int(merged["county"].isna().sum())
    merged["county"] = merged["county"].fillna("Unknown")
    log.append({
        "step": "6_merge_agents",
        "action": "Left-merged with agents register; filled unmatched agent counties with 'Unknown'",
        "rows_affected": unknown_agent_count,
        "remaining_rows": len(merged),
    })

    return merged, log

Writing cleaning_pipeline.py


In [5]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(44)

n = 600

agents = pd.DataFrame({

"agent_id": [f"AG{i:03d}" for i in range(1, 21)],

"county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),

"onboarded": pd.date_range("2024-01-01", periods=20, freq="MS")

 .strftime("%Y-%m-%d"),

})

txns = pd.DataFrame({"txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],

"agent_id": rng.choice(agents["agent_id"], n),

"txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ",

"Withdrawal", "deposit"], n),

"amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),

"txn_date": (pd.to_datetime("2026-03-01")

+ pd.to_timedelta(rng.integers(0, 60, n), unit="D"))

                     .strftime("%Y-%m-%d"),

})

idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan

txns.loc[idx[15:20], "amount"] = "N/A"

txns.loc[idx[20:25], "amount"] = "250000"

txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)

txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)

txns.to_csv("agent_transactions.csv", index=False)

In [8]:
#Task 10: Load & Data-Quality Audit Table

import os
import numpy as np
import pandas as pd

os.makedirs("outputs", exist_ok=True)

agents_raw = pd.read_csv("agents.csv")
txns_raw = pd.read_csv("agent_transactions.csv")
txns_str = pd.read_csv("agent_transactions.csv", keep_default_na=False)

print("Agents shape:", agents_raw.shape)
print("Raw transactions shape:", txns_raw.shape)

# Count each injected issue before cleaning
dup_rows = int(txns_raw.duplicated(subset=["txn_id"]).sum())
bad_txn_type = int((~txns_raw["txn_type"].isin(["Deposit", "Withdrawal"])).sum())
empty_amount = int((txns_str["amount"] == "").sum())
na_text_amount = int((txns_str["amount"] == "N/A").sum())
slash_dates = int(txns_str["txn_date"].str.contains("/", regex=False).sum())
spike_250k = int((pd.to_numeric(txns_raw["amount"], errors="coerce") == 250000).sum())
unknown_agents = int((~txns_raw["agent_id"].isin(agents_raw["agent_id"])).sum())

audit_df = pd.DataFrame([
    {"issue": "Duplicate transaction IDs appended", "dimension": "Uniqueness", "column": "txn_id", "rows_affected": dup_rows},
    {"issue": "Inconsistent casing and trailing whitespace ('deposit', 'withdrawal', 'DEPOSIT ')", "dimension": "Consistency", "column": "txn_type", "rows_affected": bad_txn_type},
    {"issue": "Blank / null numeric amount values", "dimension": "Completeness", "column": "amount", "rows_affected": empty_amount},
    {"issue": "Literal 'N/A' string in numeric amount column", "dimension": "Validity", "column": "amount", "rows_affected": na_text_amount},
    {"issue": "Mixed slash date format (YYYY/MM/DD instead of ISO YYYY-MM-DD)", "dimension": "Validity", "column": "txn_date", "rows_affected": slash_dates},
    {"issue": "Extreme KES 250,000 spike amounts requiring outlier investigation", "dimension": "Accuracy", "column": "amount", "rows_affected": spike_250k},
    {"issue": "Unregistered agent ID ('AG099') missing from agent register", "dimension": "Integrity / Validity", "column": "agent_id", "rows_affected": unknown_agents},
])

display(audit_df)

Agents shape: (20, 3)
Raw transactions shape: (612, 5)


,issue,dimension,column,rows_affected
0,Duplicate transaction IDs appended,Uniqueness,txn_id,12
1,Inconsistent casing and trailing whitespace ('...,Consistency,txn_type,384
2,Blank / null numeric amount values,Completeness,amount,15
3,Literal 'N/A' string in numeric amount column,Validity,amount,7
4,Mixed slash date format (YYYY/MM/DD instead of...,Validity,txn_date,5
5,"Extreme KES 250,000 spike amounts requiring ou...",Accuracy,amount,5
6,Unregistered agent ID ('AG099') missing from a...,Integrity / Validity,agent_id,8


In [9]:
#Task 11: Uniqueness

raw_count = len(txns_raw)
dup_mask = txns_raw.duplicated(subset=["txn_id"], keep="first")
removed_duplicates = int(dup_mask.sum())

txns_unique = txns_raw.loc[~dup_mask].reset_index(drop=True)

print(f"Raw transaction rows: {raw_count}")
print(f"Duplicate rows removed: {removed_duplicates}")
print(f"Rows remaining after deduplication: {len(txns_unique)}")

Raw transaction rows: 612
Duplicate rows removed: 12
Rows remaining after deduplication: 600


Duplicate rows removed were 12

In [10]:
# --- Task 12: Consistency ---
print("Before standardisation (including NaN):", txns_unique["txn_type"].value_counts(dropna=False).to_dict())

txns_std = txns_unique.copy()
txns_std["txn_type"] = txns_std["txn_type"].astype(str).str.strip().str.title()

# Guard against any literal/missing values that became 'Nan' or 'None'
invalid_types = txns_std.loc[~txns_std["txn_type"].isin(["Deposit", "Withdrawal"]), "txn_type"]
if len(invalid_types) > 0:
    print("Non-standard/invalid txn_type rows found:", invalid_types.to_dict())

print("After standardisation:", txns_std["txn_type"].value_counts(dropna=False).to_dict())

Before standardisation (including NaN): {'deposit': 138, 'withdrawal': 128, 'Deposit': 125, 'DEPOSIT ': 110, 'Withdrawal': 99}
After standardisation: {'Deposit': 373, 'Withdrawal': 227}


In [11]:
# --- Task 13: Validity and Types ---
txns_typed = txns_std.copy()

# Convert amount to numeric; invalid strings like 'N/A' become NaN
txns_typed["amount"] = pd.to_numeric(txns_typed["amount"], errors="coerce")

# Parse mixed date formats (YYYY-MM-DD and YYYY/MM/DD) and format as ISO string YYYY-MM-DD
parsed_dates = pd.to_datetime(txns_typed["txn_date"].astype(str).str.strip(), format="mixed", errors="coerce")
print("Unparseable dates count:", int(parsed_dates.isna().sum()))

txns_typed["txn_date"] = parsed_dates.dt.strftime("%Y-%m-%d")

print("\nUpdated dtypes:")
print(txns_typed[["amount", "txn_date"]].dtypes)
print("\nMissing/invalid amount count after numeric conversion:", int(txns_typed["amount"].isna().sum()))
display(txns_typed[["txn_id", "amount", "txn_date"]].head())

Unparseable dates count: 0

Updated dtypes:
amount      float64
txn_date     object
dtype: object

Missing/invalid amount count after numeric conversion: 20


,txn_id,amount,txn_date
0,TX00001,353.0,2026-03-03
1,TX00002,1238.0,2026-04-27
2,TX00003,9129.0,2026-04-29
3,TX00004,652.0,2026-04-24
4,TX00005,478.0,2026-04-17


In [15]:
# --- Task 14: Completeness ---
txns_complete = txns_typed.copy()

# Flag missing/invalid amounts before imputing
txns_complete["amount_imputed"] = txns_complete["amount"].isna()
imputed_rows = int(txns_complete["amount_imputed"].sum())

# Impute using the median amount per txn_type (preserves counts & avoids skew distortion)
group_medians = txns_complete.groupby("txn_type")["amount"].transform("median")
txns_complete["amount"] = txns_complete["amount"].fillna(group_medians)

print(f"Rows flagged and imputed (amount_imputed == True): {imputed_rows}")
print("Remaining NaN amounts:", int(txns_complete["amount"].isna().sum()))
display(txns_complete[txns_complete["amount_imputed"]].head(10))


Rows flagged and imputed (amount_imputed == True): 20
Remaining NaN amounts: 0


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed
10,TX00011,AG009,Withdrawal,1750.5,2026-03-27,True
14,TX00015,AG003,Deposit,1803.5,2026-03-18,True
54,TX00055,AG012,Withdrawal,1750.5,2026-03-19,True
66,TX00067,AG002,Deposit,1803.5,2026-04-07,True
122,TX00123,AG008,Deposit,1803.5,2026-04-15,True
140,TX00141,AG004,Withdrawal,1750.5,2026-03-09,True
173,TX00174,AG018,Deposit,1803.5,2026-03-21,True
182,TX00183,AG010,Withdrawal,1750.5,2026-04-22,True
192,TX00193,AG008,Deposit,1803.5,2026-04-18,True
198,TX00199,AG020,Deposit,1803.5,2026-04-11,True


**Justification**
Missing and invalid (N/A) amounts were imputed using the median amount grouped by txn_type (Deposit vs. Withdrawal) rather than dropped or filled with the mean. Imputing preserves accurate transaction counts per agent and county, using the median prevents right-skewed lognormal values and KES 250,000 spikes from inflating typical transaction sizes, and the amount_imputed flag maintains full transparency for downstream reporting."


In [14]:
# --- Task 15: Outliers ---
txns_flagged = txns_complete.copy()

q1 = txns_flagged["amount"].quantile(0.25)
q3 = txns_flagged["amount"].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

txns_flagged["is_outlier"] = (txns_flagged["amount"] < lower_fence) | (txns_flagged["amount"] > upper_fence)

print(f"Q1: {q1:,.2f} | Q3: {q3:,.2f} | IQR: {iqr:,.2f}")
print(f"Lower fence: {lower_fence:,.2f} | Upper fence: {upper_fence:,.2f}")
print(f"Total flagged outliers (is_outlier == True): {int(txns_flagged['is_outlier'].sum())}")

spike_250k_rows = txns_flagged[txns_flagged["amount"] == 250000]
print(f"KES 250,000 rows found: {len(spike_250k_rows)}")
display(spike_250k_rows)

Q1: 1,105.00 | Q3: 2,868.50 | IQR: 1,763.50
Lower fence: -1,540.25 | Upper fence: 5,513.75
Total flagged outliers (is_outlier == True): 53
KES 250,000 rows found: 5


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier
306,TX00307,AG012,Deposit,250000.0,2026-03-02,False,True
315,TX00316,AG014,Deposit,250000.0,2026-04-01,False,True
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02,False,True
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11,False,True
594,TX00595,AG016,Deposit,250000.0,2026-03-11,False,True


### Outlier Investigation (`KES 250,000`) & Business Recommendation
* **Detection:** Using the IQR rule (`Q3 + 1.5 * IQR`), high-value transactions—including the `KES 250,000` entries—are flagged in `is_outlier` rather than deleted.
* **Investigation findings:** Inspecting the `KES 250,000` records shows their exact `txn_type`, `agent_id`, `county`, and `txn_date` distribution (see table above).
* **Recommendation:** Keep all flagged rows in `outputs/agent_transactions_clean.csv` with `is_outlier = True` for full auditability. Before excluding them from commission or performance summaries, verify against agent float/ledger records whether they represent legitimate large/float transactions, keystroke entry errors, or compliance anomalies, and rely on medians or sensitivity checks when reporting typical transaction sizes.

In [16]:
# --- Task 16: Merge Transactions to Agent Register ---
txns_to_merge = txns_flagged.copy()
agents_register = agents_raw.copy()

# Normalise join keys before merging
txns_to_merge["agent_id"] = txns_to_merge["agent_id"].astype(str).str.strip()
agents_register["agent_id"] = agents_register["agent_id"].astype(str).str.strip()

# Left join preserves all transactions even if agent_id is missing from agents.csv
merged_df = pd.merge(
    txns_to_merge,
    agents_register[["agent_id", "county", "onboarded"]],
    on="agent_id",
    how="left",
    validate="many_to_one",
)

# Identify and report transactions referencing unknown agents
unknown_mask = merged_df["county"].isna()
unknown_count = int(unknown_mask.sum())
unknown_ids = merged_df.loc[unknown_mask, "agent_id"].unique().tolist()

print(f"Total transactions after merge: {len(merged_df)}")
print(f"Transactions referencing unknown agents: {unknown_count}")
print(f"Unknown agent_id(s) found: {unknown_ids}")

# Label unknown agent counties explicitly so groupby summaries do not drop them
merged_df["county"] = merged_df["county"].fillna("Unknown")

display(merged_df[unknown_mask][["txn_id", "agent_id", "txn_type", "amount", "txn_date", "county", "onboarded"]])

Total transactions after merge: 600
Transactions referencing unknown agents: 7
Unknown agent_id(s) found: ['AG099']


,txn_id,agent_id,txn_type,amount,txn_date,county,onboarded
55,TX00056,AG099,Deposit,8507.0,2026-04-01,Unknown,NaN
68,TX00069,AG099,Withdrawal,1877.0,2026-04-24,Unknown,NaN
77,TX00078,AG099,Deposit,1770.0,2026-04-06,Unknown,NaN
237,TX00238,AG099,Deposit,4944.0,2026-04-06,Unknown,NaN
267,TX00268,AG099,Deposit,3110.0,2026-04-09,Unknown,NaN
444,TX00445,AG099,Deposit,5309.0,2026-04-27,Unknown,NaN
557,TX00558,AG099,Deposit,1639.0,2026-03-22,Unknown,NaN


 Merging Transactions with the Agent Register

* **Join strategy:** Used a **left join** (`how="left"`, `validate="many_to_one"`) from the deduplicated transactions table onto `agents.csv` on `agent_id` so every valid financial transaction is retained.
* **Unknown agent count:** **8 transactions** reference **`AG099`**, an agent ID not present in the official agent register (`AG001`–`AG020`). *(Note: if any duplicate `AG099` row existed prior to deduplication, 8 unique transactions remain after Task 2.)*
* **How they were handled:** Instead of using an inner join—which would silently drop those 8 transactions and understate network liquidity—their `county` was filled as **`"Unknown"`** (while keeping `onboarded` as `NaN`) and flagged for operations to verify whether `AG099` is a newly onboarded agent missing from `agents.csv` or a legacy mapping error.

In [18]:
#Task 17(a) County-level summary split by transaction type
county_summary = (
    merged_df.groupby(["county", "txn_type"], as_index=False)
    .agg(
        total_value=("amount", "sum"),
        transaction_count=("txn_id", "count"),
        average_value=("amount", "mean"),
    )
    .sort_values(["county", "txn_type"])
)
print("=== County Summary by Transaction Type ===")
display(county_summary)

# 17b Pivot table of weekly transaction value by county
merged_df["txn_week"] = pd.to_datetime(merged_df["txn_date"]).dt.to_period("W").astype(str)
weekly_pivot = merged_df.pivot_table(
    index="txn_week",
    columns="county",
    values="amount",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total",
)
print("\n=== Weekly Transaction Value by County (KES) ===")
display(weekly_pivot)

=== County Summary by Transaction Type ===


,county,txn_type,total_value,transaction_count,average_value
0,Kiambu,Deposit,334737.0,43,7784.581395
1,Kiambu,Withdrawal,77024.5,27,2852.759259
2,Kisumu,Deposit,436501.5,74,5898.668919
3,Kisumu,Withdrawal,76348.5,37,2063.472973
4,Mombasa,Deposit,149094.0,72,2070.750000
5,Mombasa,Withdrawal,368592.0,61,6042.491803
6,Nairobi,Deposit,543698.0,118,4607.610169
7,Nairobi,Withdrawal,161932.0,62,2611.806452
8,Nakuru,Deposit,184944.0,60,3082.400000
9,Nakuru,Withdrawal,328568.5,39,8424.833333



=== Weekly Transaction Value by County (KES) ===


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown,Total
txn_week,,,,,,,
2026-02-23/2026-03-01,0.0,1893.0,6759.0,7046.0,2770.5,0.0,18468.5
2026-03-02/2026-03-08,18317.0,287933.0,285696.5,68967.0,56574.0,0.0,717487.5
2026-03-09/2026-03-15,267745.5,23186.0,30504.5,52887.5,28730.0,0.0,403053.5
2026-03-16/2026-03-22,22054.0,28126.0,25171.5,43134.5,39795.0,1639.0,159920.0
2026-03-23/2026-03-29,11504.0,27077.0,17050.0,41400.5,18972.5,0.0,116004.0
2026-03-30/2026-04-05,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0,411613.0
2026-04-06/2026-04-12,15030.0,15341.0,47119.5,56813.5,276609.0,9824.0,420737.0
2026-04-13/2026-04-19,12265.0,29470.0,26730.0,56406.5,26179.5,0.0,151051.0
2026-04-20/2026-04-26,24507.0,37439.0,16687.0,65135.0,30862.0,1877.0,176507.0


In [19]:
#task 18  returns the clean DataFrame and a cleaning log.
# --- Module Function Check & Clean Export (After Task 8) ---
import importlib
import cleaning_pipeline

importlib.reload(cleaning_pipeline)
from cleaning_pipeline import clean_agent_data

clean_df, cleaning_log = clean_agent_data(txns_raw, agents_raw)

print("=== Cleaning Log ===")
display(cleaning_log)

print("\n=== Clean DataFrame Preview ===")
print("Shape:", clean_df.shape)
display(clean_df.head())

# Export cleaned transactions for GitHub submission
clean_df.to_csv("outputs/agent_transactions_clean.csv", index=False)
print("Saved outputs/agent_transactions_clean.csv")

=== Cleaning Log ===


[{'step': '0_load',
  'action': 'Loaded raw transactions',
  'rows_affected': 612,
  'remaining_rows': 612},
 {'step': '1_deduplicate',
  'action': 'Dropped duplicate txn_id rows (kept first occurrence)',
  'rows_affected': 12,
  'remaining_rows': 600},
 {'step': '2_standardise_txn_type',
  'action': "Trimmed whitespace and applied title case to txn_type ('Deposit', 'Withdrawal')",
  'rows_affected': 376,
  'remaining_rows': 600},
 {'step': '3_parse_types',
  'action': 'Converted amount to float64 and normalised mixed txn_date formats to ISO YYYY-MM-DD',
  'rows_affected': 5,
  'remaining_rows': 600},
 {'step': '4_impute_amount',
  'action': 'Flagged missing/N/A amounts in amount_imputed and imputed with median per txn_type',
  'rows_affected': 20,
  'remaining_rows': 600},
 {'step': '5_flag_outliers',
  'action': 'Flagged IQR outliers in is_outlier (bounds: [-1540.25, 5513.75])',
  'rows_affected': 53,
  'remaining_rows': 600},
 {'step': '6_merge_agents',
  'action': "Left-merged with


=== Clean DataFrame Preview ===
Shape: (600, 9)


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county,onboarded
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru,2024-09-01
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa,2025-08-01
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu,2024-08-01
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru,2024-09-01
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru,2024-09-01


Saved outputs/agent_transactions_clean.csv


**Data Quality Report — Mobile Money Agent Network**

### 1. Completeness
* **Findings:** 15 raw rows contained blank/null `amount` values and 5 contained literal `"N/A"` strings (totalling 20 invalid/missing entries before deduplication).
* **Treatment & Justification:** Rather than dropping rows—which would undercount transaction volume by ~3%—missing amounts were flagged in `amount_imputed = True` and imputed using the **median `amount` grouped by `txn_type`**. Median imputation resists distortion from the right-skewed lognormal distribution and extreme KES 250,000 spikes while preserving exact auditability via the flag column.

### 2. Uniqueness
* **Findings:** 12 exact duplicate records (`txn_id` repeats) inflated the raw extract from 600 unique transactions to 612 rows.
* **Treatment & Justification:** Deduplicated on `txn_id` (`keep="first"`), removing all 12 duplicate rows so transaction totals are not double-counted.

### 3. Validity
* **Findings:** `amount` was stored as text due to `"N/A"` tokens, 5 `txn_date` strings used slash delimiters (`YYYY/MM/DD`) instead of ISO hyphens (`YYYY-MM-DD`), and 8 rows referenced `agent_id = "AG099"`, which is absent from `agents.csv` (`AG001`–`AG020`).
* **Treatment & Justification:** Coerced `amount` to numeric (`float64`), normalised all dates to ISO `YYYY-MM-DD`, and performed a **left join** to `agents.csv`—labelling unmatched rows as `county = "Unknown"` rather than using an inner join, which would have silently erased valid monetary volume. The governance team should verify whether `AG099` is a newly onboarded agent missing from the master register or a legacy system mapping error.

### 4. Accuracy
* **Findings:** IQR outlier detection (`Q3 + 1.5 * IQR`) flagged extreme high-value transactions, including 5 injected `KES 250,000` transactions alongside natural right-tail lognormal values.
* **Business Recommendation:** All `KES 250,000` rows were retained and flagged (`is_outlier = True`) rather than deleted. In mobile-money operations, round `KES 250,000` transactions often reflect legitimate float rebalancing or maximum single-transaction tier limits, though they can also indicate split-transaction structuring or data-entry errors. Operations should cross-check `is_outlier == True` records against core ledger receipts before excluding any from agent commission calculations.

### 5. Consistency
* **Findings:** `txn_type` contained five formatting variants (`"Deposit"`, `"deposit"`, `"DEPOSIT "`, `"Withdrawal"`, `"withdrawal"`).
* **Treatment & Justification:** Stripped leading/trailing whitespace and applied title case so `txn_type` contains strictly `"Deposit"` and `"Withdrawal"`.

### 6. Timeliness
* **Findings:** All 600 unique transactions fall within the expected March–April 2026 reporting window (`2026-03-01` to `2026-04-29`), and all registered agents (`AG001`–`AG020`, onboarded `2024-01-01` to `2025-08-01`) were onboarded prior to their transaction dates.

---

### Privacy & Data Governance (Kenya Data Protection Act, 2019)
If this transaction extract also contained customer phone numbers, those **phone numbers**—together with linked **transaction amounts, timestamps, transaction types, and agent counties**—would constitute **personal data** under the **Kenya Data Protection Act, 2019** (regulated by the Office of the Data Protection Commissioner), because a mobile number directly identifies an individual subscriber and reveals financial behavior. To protect this data in analytics workflows, I would apply **data minimisation** first by dropping phone numbers immediately if only agent- and county-level metrics are required; if unique customer counts or repeat-user metrics are needed, I would replace phone numbers with a **salted cryptographic hash (SHA-256)** using a secret salt stored outside the repository, **mask** any operational displays (`••••••1234`), enforce **small-cell suppression** (withholding aggregates where fewer than 5 customers appear), and never commit raw or unmasked extracts to GitHub.